In [ ]:
import os

%load_ext autoreload
%autoreload 2

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import deephit_cancer_comparison.constants as const
from deephit_cancer_comparison.constants import GRAPH_PATH, SURVSHAP_PATH

EVAL_TIMES = [const.TIMESTEP * i for i in range(1, const.T_MAX // const.TIMESTEP + 1)]

TOP_K = 15

cancer_types = [
    "breast",
    "corpus",
    "kidney_parenchyma",
    "lung_and_bronchus",
    "melanoma_of_the_skin",
    "pancreas",
    "prostate",
    "thyroid",
    "urinary_bladder",
    "colon_and_rectum",
]

for cancer_type in cancer_types:
    if not os.path.exists(GRAPH_PATH / cancer_type):
        os.makedirs(GRAPH_PATH / cancer_type, exist_ok=True)

COHORT = cancer_types[7]

In [ ]:
def load_cohort(cohort: str, root: Path = SURVSHAP_PATH):
    """Load aggregated (scalar per feature per obs) and time-varying SurvSHAP results."""
    cohort_dir = root / cohort
    scalar_df = pd.read_parquet(cohort_dir / "survshap_aggregated.parquet")
    tv_df = pd.read_parquet(cohort_dir / "survshap_timevarying.parquet")
    return scalar_df, tv_df

In [ ]:
def plot_top_features(scalar_df: pd.DataFrame, cohort: str, top_k: int = 15, figsize=(8, 6)):
    summary = (
        scalar_df.groupby("variable_name")["aggregated_change"]
        .agg(median="median", q25=lambda s: s.quantile(0.25), q75=lambda s: s.quantile(0.75))
        .sort_values("median", ascending=False)
        .head(top_k)
    )

    fig, ax = plt.subplots(figsize=figsize)
    y = np.arange(len(summary))
    ax.barh(
        y,
        summary["median"],
        xerr=[summary["median"] - summary["q25"], summary["q75"] - summary["median"]],
        color="steelblue",
        alpha=0.85,
    )
    ax.set_yticks(y)
    ax.set_yticklabels(summary.index)
    ax.invert_yaxis()
    ax.set_xlabel("Aggregated |SurvSHAP(t)| (median across patients, IQR bars)")
    ax.set_title(f"Top {top_k} features — {cohort}")
    plt.tight_layout()
    return summary


# summary = plot_top_features(scalar_df, COHORT, top_k=TOP_K)
# summary

In [ ]:
def plot_timevarying(
    tv_df: pd.DataFrame, feature: str, cohort: str, show_individuals: bool = True, figsize=(10, 5)
):
    sub = tv_df[tv_df["variable_name"] == feature]
    if sub.empty:
        raise ValueError(
            f"No data for feature '{feature}'. "
            f"Available: {sorted(tv_df['variable_name'].unique())[:10]}..."
        )

    time_cols = [c for c in sub.columns if isinstance(c, str) and c.startswith("t = ")]
    times = np.array([float(c.removeprefix("t = ")) for c in time_cols])
    values = sub[time_cols].to_numpy()

    fig, ax = plt.subplots(figsize=figsize)
    if show_individuals:
        for row in values:
            ax.plot(times, row, color="steelblue", alpha=0.12, linewidth=0.7)
    ax.plot(
        times, values.mean(axis=0), color="darkblue", linewidth=2.2, label=f"Mean (n={len(values)})"
    )
    ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
    ax.set_xlabel("Time (DeepHit bin index)")
    ax.set_ylabel(r"SurvSHAP$_t(x, d)$")
    ax.set_title(f"Time-varying attribution of '{feature}' — {cohort}")
    ax.legend()
    plt.tight_layout()


# FEATURE_TO_PLOT = summary.index[0]
# plot_timevarying(tv_df, feature=FEATURE_TO_PLOT, cohort=COHORT)

In [ ]:
def plot_value_vs_shap(scalar_df: pd.DataFrame, feature: str, cohort: str, figsize=(7, 5)):
    sub = scalar_df[scalar_df["variable_name"] == feature]
    if sub.empty:
        raise ValueError(f"No data for feature '{feature}'")

    fig, ax = plt.subplots(figsize=figsize)
    ax.scatter(sub["variable_value"], sub["aggregated_change"], alpha=0.5, s=20, color="steelblue")
    ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
    ax.set_xlabel(f"{feature} (observed value)")
    ax.set_ylabel("Aggregated |SurvSHAP(t)|")
    ax.set_title(f"Feature value vs. attribution — '{feature}' in {cohort}")
    plt.tight_layout()


# plot_value_vs_shap(scalar_df, feature=FEATURE_TO_PLOT, cohort=COHORT)